### Word Document Processing

In [19]:
from langchain_community.document_loaders import Docx2txtLoader,UnstructuredWordDocumentLoader


In [20]:
## MEthod1: Using Docx2txtLoader
print("1️⃣ Using Docx2txtLoader")
try:
    docx_loader=Docx2txtLoader("data/word_files/passportfaq.docx")
    docs=docx_loader.load()
    print(f"✅ Loaded {len(docs)} document(s)")
    print(f"Content preview: {docs[0].page_content[:200]}...")
    print(f"Metadata: {docs[0].metadata}")

except Exception as e:
    print(f"Error: {e}")

1️⃣ Using Docx2txtLoader
Error: No module named 'docx2txt'


In [21]:
## MEthod2
print("\n2️⃣ Using UnstructuredWordDocumentLoader")

try:
    unstructured_loader=UnstructuredWordDocumentLoader("data/word_files/passportfaq.docx",mode="elements")
    unstructured_docs=unstructured_loader.load()
    
    print(f"✅ Loaded {len(unstructured_docs)} elements")
    for i, doc in enumerate(unstructured_docs[:3]):
        print(f"\nElement {i+1}:")
        print(f"Type: {doc.metadata.get('category', 'unknown')}")
        print(f"Content: {doc.page_content[:100]}...")


except Exception as e:
    print(e) 



2️⃣ Using UnstructuredWordDocumentLoader
unstructured package not found, please install it with `pip install unstructured`


In [22]:
# unstructured_docs

In [23]:
from docx import Document
from langchain_core.documents import Document as LCDocument
from typing import List

SECTION_HEADINGS = {
    "account", "account and settings",
    "appointments",
    "payment",
    "application",
    "general queries",
    "further questions",
    "others"
}

def load_passport_faq_docx(path: str) -> List[LCDocument]:
    doc = Document(path)

    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

    faq_chunks = []
    buffer = []
    current_category = None

    for para in paragraphs:
        normalized = para.lower()

        # 1️⃣ Detect section headings
        if normalized in SECTION_HEADINGS:
            current_category = para
            buffer = []  # clear buffer so headings don't become chunks
            continue

        # 2️⃣ Detect start of a NEW FAQ (English question line)
        is_english_question = para.endswith("?") and para.isascii()

        if is_english_question and buffer:
            faq_chunks.append({
                "text": "\n".join(buffer),
                "category": current_category
            })
            buffer = [para]
        else:
            buffer.append(para)

    # Add last FAQ
    if buffer:
        faq_chunks.append({
            "text": "\n".join(buffer),
            "category": current_category
        })

    documents = []
    for i, faq in enumerate(faq_chunks):
        documents.append(
            LCDocument(
                page_content=faq["text"],
                metadata={
                    "source": "Passport FAQ DOCX",
                    "doc_type": "passport_faq",
                    "faq_category": faq["category"],
                    "chunk_id": i
                }
            )
        )

    return documents


In [24]:
docs = load_passport_faq_docx("data/word_files/passportfaq.docx")


print(f"Loaded {len(docs)} FAQ chunks")
print(docs[0].page_content[:500])
print(docs[0].metadata)


Loaded 83 FAQ chunks
I forgot the password of my online application account – what should I do?
I forgot the password of my online application account – what should I do?
আমার অনলাইন আবেদনের পাসওয়ার্ড ভুলে গেছি। এখন কি করা উচিত?
Last updated: 30 August 2021
In case you forgot the password of your online application account you can use the password recovery function.
Go to the account sign in
Click the “Forgot password” link
Enter your email address
Check your email account for the recovery message and follow the ins
{'source': 'Passport FAQ DOCX', 'doc_type': 'passport_faq', 'faq_category': 'Account and Settings', 'chunk_id': 0}
